# Two-Stage Information Retrieval Pipeline

Pipeline de recuperação de informação em 2 estágios para documentos em português:
1. **Stage 1 — Dense Retrieval:** Bi-encoder multilingual embeda query e corpus em um espaço vetorial compartilhado. Busca por similaridade no Chroma (maximiza **recall**).
2. **Stage 2 — Cross-Encoder Reranking:** Cross-encoder multilingual pontua pares (query, passagem) conjuntamente. Reordena os candidatos (maximiza **precisão**).

**Características:**
- Arquitetura modular com `dataclass` para configuração
- Embedder e reranker **multilingual** (otimizados para PT-BR)
- Persistência via ChromaDB com IDs determinísticos
- Chunking baseado em tokenizer do HuggingFace
- Avaliação quantitativa com MRR@k e Recall@k usando gold standard

In [1]:
# Descomente para instalar dependências, se necessário
# !pip install langchain-community langchain-huggingface langchain-chroma langchain-text-splitters
# !pip install sentence-transformers pypdf transformers

## 2. Imports e Configuração

In [2]:
from __future__ import annotations

import hashlib
import re
from dataclasses import dataclass, field
from typing import List, Optional, Sequence, Dict, Any

import torch
from sentence_transformers import CrossEncoder

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from transformers import AutoTokenizer

/home/senacgoon.local/202473567/Documents/senac_ia/uc15/senac_ia_uc15_nlp_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
@dataclass
class PipelineConfig:
    """Configuração centralizada do pipeline de IR."""

    # Caminhos
    pdf_path: str = "/home/senacgoon.local/202473567/Documents/senac_ia/uc15/senac_ia_uc15_nlp_project/data/pablo/sindilojas_2025_2026.pdf"
    gold_standard_path: str = "/home/senacgoon.local/202473567/Documents/senac_ia/uc15/senac_ia_uc15_nlp_project/data/pablo/gold_standard.md"
    persist_directory: str = "./chroma_langchain_db_v2"
    collection_name: str = "sindilojas_2stage"

    # Modelos — multilingual para PT-BR
    embedding_model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    reranker_model_name: str = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

    # Chunking
    chunk_size: int = 200
    chunk_overlap: int = 30

    # Retrieval
    initial_k: int = 10
    final_k: int = 5

    # Performance
    device: Optional[str] = None
    normalize_embeddings: bool = True
    reranker_batch_size: int = 8


def get_device(explicit: Optional[str] = None) -> str:
    if explicit:
        return explicit
    return "cuda" if torch.cuda.is_available() else "cpu"


config = PipelineConfig()
device = get_device(config.device)
print(f"Device: {device}")
print(f"Embedding model: {config.embedding_model_name}")
print(f"Reranker model: {config.reranker_model_name}")

Device: cpu
Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Reranker model: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


## 3. Carregamento e Chunking do Documento

Usa `PyPDFLoader` para carregar o PDF e `RecursiveCharacterTextSplitter.from_huggingface_tokenizer` para chunking baseado em tokens (não caracteres). Isso garante que os chunks respeitem os limites de sequência do modelo de embedding.

In [4]:
# Carregar PDF
loader = PyPDFLoader(config.pdf_path)
docs = loader.load()
print(f"Páginas carregadas: {len(docs)}")

# Carregar tokenizer do modelo de embedding para chunking preciso em tokens
embedding_tokenizer = AutoTokenizer.from_pretrained(config.embedding_model_name)

text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=embedding_tokenizer,
    chunk_size=config.chunk_size,
    chunk_overlap=config.chunk_overlap,
    add_start_index=True,
)

all_splits = text_splitter.split_documents(docs)
print(f"Total de chunks: {len(all_splits)}")
print(f"Exemplo de chunk (primeiros 200 chars): {all_splits[0].page_content[:200]}")
print(f"Metadata: {all_splits[0].metadata}")

Páginas carregadas: 38


Total de chunks: 188
Exemplo de chunk (primeiros 200 chars): Sindicato dos Comerciários de São Paulo 
Rua Formosa, 99 Centro 
CEP 01049-000 - São Paulo - SP 
Fone:. 2121-5900 
e-mail: 
atendimento@comerciarios.org.br  
Sindicato do Comércio Varejista e Lojista 
Metadata: {'producer': 'Microsoft® Word para Microsoft 365 - by Lacuna Software PKI SDK', 'creator': 'Microsoft® Word para Microsoft 365', 'creationdate': '2025-11-24T18:02:52-03:00', 'documentkey': '5ccd231f-5127-4696-84df-b576074c6eba', 'author': 'Vivianne Morais', 'moddate': '2025-11-24T21:54:55+00:00', 'source': '/home/senacgoon.local/202473567/Documents/senac_ia/uc15/senac_ia_uc15_nlp_project/data/pablo/sindilojas_2025_2026.pdf', 'total_pages': 38, 'page': 0, 'page_label': '1', 'start_index': 0}


## 4. Embedding Model + ChromaDB com IDs Determinísticos

IDs determinísticos (SHA-256 de source + page + content) garantem que re-execuções não dupliquem dados no vector store.

In [5]:
def deterministic_chunk_id(source: str, page: int, chunk_text: str) -> str:
    """Gera ID determinístico para evitar duplicação no vector store."""
    payload = f"{source}|{page}|{chunk_text.strip().lower()}"
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


# Inicializar embedding model multilingual
embeddings = HuggingFaceEmbeddings(
    model_name=config.embedding_model_name,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": config.normalize_embeddings},
)

# Verificar dimensionalidade
test_vec = embeddings.embed_query("teste")
print(f"Dimensão dos embeddings: {len(test_vec)}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7153.90it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dimensão dos embeddings: 384


In [6]:
# Inicializar ChromaDB
vector_store = Chroma(
    collection_name=config.collection_name,
    embedding_function=embeddings,
    persist_directory=config.persist_directory,
)

# Gerar IDs determinísticos e ingerir chunks
chunk_ids = [
    deterministic_chunk_id(
        chunk.metadata.get("source", ""),
        chunk.metadata.get("page", 0),
        chunk.page_content,
    )
    for chunk in all_splits
]

ids = vector_store.add_documents(documents=all_splits, ids=chunk_ids)
print(f"Documentos indexados: {len(ids)}")
print(f"Collection count: {vector_store._collection.count()}")

Documentos indexados: 188
Collection count: 188


## 5. Two-Stage Retriever

Classe modular que encapsula o pipeline completo:
- `_dense_retrieve()`: busca os top-k candidatos via similaridade vetorial (Stage 1)
- `_rerank()`: re-pontua os candidatos com cross-encoder e retorna os top-n finais (Stage 2)
- `invoke()` / `batch()`: interface compatível com LangChain LCEL

In [7]:
class TwoStageRetriever:
    """
    Pipeline de retrieval em 2 estágios:
    Stage 1 — Dense retrieval via Chroma (bi-encoder)
    Stage 2 — Cross-encoder reranking (sentence-transformers)
    """

    def __init__(
        self,
        vectorstore: Chroma,
        reranker_model_name: str,
        initial_k: int = 20,
        final_k: int = 5,
        device: Optional[str] = None,
        reranker_batch_size: int = 8,
    ) -> None:
        self.vectorstore = vectorstore
        self.initial_k = initial_k
        self.final_k = final_k
        self.device = get_device(device)
        self.reranker_batch_size = reranker_batch_size

        self.cross_encoder = CrossEncoder(
            reranker_model_name,
            device=self.device,
        )

    def _dense_retrieve(self, query: str) -> List[Document]:
        """Stage 1: busca os initial_k candidatos mais similares."""
        return self.vectorstore.similarity_search(query, k=self.initial_k)

    def _rerank(self, query: str, docs: Sequence[Document]) -> List[Document]:
        """Stage 2: re-pontua com cross-encoder e retorna top final_k."""
        if not docs:
            return []

        pairs = [(query, doc.page_content) for doc in docs]
        scores = self.cross_encoder.predict(
            pairs,
            batch_size=self.reranker_batch_size,
            show_progress_bar=False,
        )

        scored_docs = []
        for doc, score in zip(docs, scores):
            metadata = dict(doc.metadata) if doc.metadata else {}
            metadata["reranker_score"] = float(score)
            scored_docs.append(Document(page_content=doc.page_content, metadata=metadata))

        scored_docs.sort(key=lambda d: d.metadata["reranker_score"], reverse=True)
        return scored_docs[: self.final_k]

    def invoke(self, query: str) -> List[Document]:
        """Executa o pipeline completo para uma query."""
        candidates = self._dense_retrieve(query)
        return self._rerank(query, candidates)

    def batch(self, queries: Sequence[str]) -> List[List[Document]]:
        """Executa o pipeline para múltiplas queries."""
        return [self.invoke(q) for q in queries]

    def as_runnable(self) -> RunnableLambda:
        """Retorna um Runnable compatível com LangChain LCEL."""
        return RunnableLambda(self.invoke)

In [8]:
# Instanciar o retriever de 2 estágios
retriever = TwoStageRetriever(
    vectorstore=vector_store,
    reranker_model_name=config.reranker_model_name,
    initial_k=config.initial_k,
    final_k=config.final_k,
    device=config.device,
    reranker_batch_size=config.reranker_batch_size,
)
print("Retriever inicializado.")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5670.46it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retriever inicializado.


## 6. Teste Rápido — Busca com Reranking

Comparação visual dos resultados do Stage 1 (dense only) vs Stage 2 (após reranking).

In [9]:
def print_results(docs: Sequence[Document], max_chars: int = 300) -> None:
    """Imprime resultados rankeados com score e trecho do conteúdo."""
    for i, doc in enumerate(docs, 1):
        score = doc.metadata.get("reranker_score", None)
        source = doc.metadata.get("source", "N/A")
        page = doc.metadata.get("page", "?")

        print("=" * 80)
        print(f"  Rank {i} | Página {page} | Source: {source}")
        if score is not None:
            print(f"  Reranker score: {score:.4f}")
        print("-" * 80)
        print(f"  {doc.page_content[:max_chars].strip()}")
        print()


# Query de teste
query = "Quais as garantias ao retornar de férias?"

print(">>> Stage 1 — Dense Retrieval (top 5):")
print()
stage1_docs = retriever._dense_retrieve(query)
for i, doc in enumerate(stage1_docs[:5], 1):
    print(f"  [{i}] {doc.page_content[:120].strip()}")
    print()

print("\n>>> Stage 2 — Após Cross-Encoder Reranking:")
print()
results = retriever.invoke(query)
print_results(results)

>>> Stage 1 — Dense Retrieval (top 5):

  [1] Parágrafo único - A garantia prevista nesta cláusula poderá ser substituída por indenização  
correspondente aos dias ai

  [2] sábados, domingos, feriados, ou dias já compensados, sendo veda da a concessão das férias 
individuais no período de 2 (

  [3] O período das férias será computado, para todos os efeitos, como tempo de serviço. 
 
III - JORNADA REDUZIDA -  Consider

  [4] computados nessa garantia os 2 dias de folga previstos na letra “f” da cláusula 41. 
 
Parágrafo 2º  - A garantia previs

  [5] e) ressarcimento de despesas com transporte, de ida e volta, sem  nenhum ônus ou desconto 
para o empregado; 
 
f) para


>>> Stage 2 — Após Cross-Encoder Reranking:

  Rank 1 | Página 20 | Source: /home/senacgoon.local/202473567/Documents/senac_ia/uc15/senac_ia_uc15_nlp_project/data/pablo/sindilojas_2025_2026.pdf
  Reranker score: 1.2312
--------------------------------------------------------------------------------
  Parágrafo único - 

## 7. Avaliação Quantitativa — MRR@k e Recall@k

Avalia o pipeline usando o **gold standard** (10 perguntas com respostas esperadas).

- **MRR@k** (Mean Reciprocal Rank): média de 1/posição do primeiro resultado relevante. Quanto mais perto de 1.0, melhor.
- **Recall@k**: fração das queries em que a resposta correta aparece em algum dos top-k resultados.

In [10]:
def parse_gold_standard(filepath: str) -> List[Dict[str, str]]:
    """Parseia o gold standard em uma lista de dicts com 'pergunta' e 'resposta'."""
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    entries = []
    blocks = re.split(r"Questão \d+:", content)
    for block in blocks:
        block = block.strip()
        if not block:
            continue

        match_p = re.search(r"Pergunta:\s*(.+?)(?:\n|$)", block)
        match_r = re.search(r"Resposta:\s*(.+?)(?:\n|$)", block)
        if match_p and match_r:
            entries.append({
                "pergunta": match_p.group(1).strip(),
                "resposta": match_r.group(1).strip(),
            })
    return entries


gold = parse_gold_standard(config.gold_standard_path)
print(f"Gold standard carregado: {len(gold)} questões")
for i, g in enumerate(gold, 1):
    print(f"  Q{i}: {g['pergunta'][:80]}...")

Gold standard carregado: 10 questões
  Q1: A partir de quando os salários fixos ou parte fixa dos salários serão reajustado...
  Q2: Qual o valor mínimo de contribuição deverá ser recolhido caso a filial sem capit...
  Q3: Quais dias as férias individuais ou coletivas não poderão coincidir?...
  Q4: A partir de quando os salários fixos ou parte fixa dos salários serão reajustado...
  Q5: Quais as exceções aonde não é autorizado o trabalho aos feriados?...
  Q6: Por qual prazo fica assegurado a manutenção do contrato de trabalho do empregado...
  Q7: Qual o artigo da CLT que autoriza a prática da jornada 12x36?...
  Q8: Qual intervalo mínimo deverá ser assegurado aos empregados?...
  Q9: Qual o valor máximo do desconto do vale-transporte?...
  Q10: Qual o valor da multa caso descumprimento de qualquer cláusula a favor do empreg...


In [11]:
def is_relevant(doc_content: str, expected_answer: str, threshold: float = 0.3) -> bool:
    """
    Verifica se o conteúdo do documento contém informação relevante à resposta esperada.
    Usa overlap de palavras-chave significativas (>3 chars) como heurística.
    """
    def normalize(text: str) -> set:
        words = re.findall(r"\w+", text.lower())
        return {w for w in words if len(w) > 3}

    doc_words = normalize(doc_content)
    answer_words = normalize(expected_answer)

    if not answer_words:
        return False

    overlap = len(doc_words & answer_words) / len(answer_words)
    return overlap >= threshold


def evaluate_retriever(
    retriever_fn,
    gold_standard: List[Dict[str, str]],
    k: int = 5,
) -> Dict[str, float]:
    """
    Avalia um retriever com MRR@k e Recall@k.

    retriever_fn: callable que recebe uma query e retorna List[Document]
    """
    reciprocal_ranks = []
    recalls = []

    for entry in gold_standard:
        query = entry["pergunta"]
        expected = entry["resposta"]
        results = retriever_fn(query)[:k]

        # Encontrar a posição do primeiro resultado relevante
        found_rank = 0
        for rank, doc in enumerate(results, 1):
            if is_relevant(doc.page_content, expected):
                found_rank = rank
                break

        reciprocal_ranks.append(1.0 / found_rank if found_rank > 0 else 0.0)
        recalls.append(1.0 if found_rank > 0 else 0.0)

    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)
    recall = sum(recalls) / len(recalls)

    return {"MRR@k": mrr, "Recall@k": recall, "k": k}

In [12]:
# Avaliar Stage 1 (dense only) vs Stage 2 (dense + rerank)

def stage1_only(query: str) -> List[Document]:
    """Retrieval apenas com bi-encoder (sem reranking)."""
    return vector_store.similarity_search(query, k=config.final_k)


print("Avaliando Stage 1 (Dense Retrieval apenas)...")
metrics_stage1 = evaluate_retriever(stage1_only, gold, k=config.final_k)
print(f"  MRR@{metrics_stage1['k']}: {metrics_stage1['MRR@k']:.4f}")
print(f"  Recall@{metrics_stage1['k']}: {metrics_stage1['Recall@k']:.4f}")

print()

print("Avaliando Stage 2 (Dense + Cross-Encoder Reranking)...")
metrics_stage2 = evaluate_retriever(retriever.invoke, gold, k=config.final_k)
print(f"  MRR@{metrics_stage2['k']}: {metrics_stage2['MRR@k']:.4f}")
print(f"  Recall@{metrics_stage2['k']}: {metrics_stage2['Recall@k']:.4f}")

print()
print("=" * 50)
delta_mrr = metrics_stage2["MRR@k"] - metrics_stage1["MRR@k"]
delta_recall = metrics_stage2["Recall@k"] - metrics_stage1["Recall@k"]
print(f"Melhoria com reranking:")
print(f"  MRR:    {delta_mrr:+.4f}")
print(f"  Recall: {delta_recall:+.4f}")

Avaliando Stage 1 (Dense Retrieval apenas)...
  MRR@5: 0.7033
  Recall@5: 0.9000

Avaliando Stage 2 (Dense + Cross-Encoder Reranking)...
  MRR@5: 0.9000
  Recall@5: 0.9000

Melhoria com reranking:
  MRR:    +0.1967
  Recall: +0.0000


## 8. Avaliação Detalhada por Questão

Visualiza o resultado do retriever para cada pergunta do gold standard, mostrando se o trecho relevante foi recuperado e em qual posição.

In [13]:
for i, entry in enumerate(gold, 1):
    query = entry["pergunta"]
    expected = entry["resposta"]
    results = retriever.invoke(query)

    print(f"{'=' * 80}")
    print(f"Q{i}: {query}")
    print(f"Resposta esperada: {expected[:100]}...")
    print()

    found = False
    for rank, doc in enumerate(results, 1):
        relevant = is_relevant(doc.page_content, expected)
        marker = ">>>" if relevant else "   "
        score = doc.metadata.get("reranker_score", 0)
        if relevant:
            found = True
        print(f"  {marker} Rank {rank} (score: {score:.4f}) | p.{doc.metadata.get('page', '?')}")
        print(f"      {doc.page_content[:150].strip()}")
        print()

    status = "ENCONTRADA" if found else "NAO ENCONTRADA"
    print(f"  Resultado: {status}")
    print()

Q1: A partir de quando os salários fixos ou parte fixa dos salários serão reajustados?
Resposta esperada: Os salários fixos ou parte fixa dos salários mistos serão reajustados a partir de 01 de setembro de ...

  >>> Rank 1 (score: 0.5401) | p.1
      os salários dos meses de competência JANEIRO e FEVEREIRO de 2026. 
 
Parágrafo 6º  - As empresas que optarem pelo parcelamento deverão observar para o

  >>> Rank 2 (score: 0.0816) | p.0
      índice de 6,00% (seis por cento)  incidente sobre os salários já reajustados na Convenção 
Coletiva de Trabalho 2024/2025, observada a cláusula “ EMPR

  >>> Rank 3 (score: -0.8957) | p.9
      itens “a”, “b” e “c” serão proporcionais à respectiva jornada. 
 
Parágrafo 8º - O piso salarial previsto no item “a” desta cláusula está sujeito a al

  >>> Rank 4 (score: -1.3912) | p.2
      de natureza trabalhista, previdenciária e tributária incidentes serão recolhido s na mesma 
época do pagamento das diferenças salariais. 
 
Parágrafo

  >>> Rank 5 (sc